<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/00_introduction/05_pagerank_motivation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)



# 누구나 이해하는 페이지랭크<br>PageRank for Everyone

> 이 노트북은 중·고등학생도 따라올 수 있도록 쓴 *입문용* 이다. 행렬도, 고유치도, 어려운 기호도 쓰지 않는다. 같은 아이디어를 수학으로 정리한 것은 **구글 페이지랭크 (Google PageRank)** 노트북에 있다.<br>
> This is an *introductory* version a middle- or high-school student can follow — no matrices, no eigenvalues, no scary symbols. The same idea written in mathematics is in the **Google PageRank** notebook.


## 문제<br>The problem

인터넷에는 수십억 개의 웹페이지가 있다. "강아지" 를 검색하면 수천 개가 걸린다. 그 중 어떤 페이지를 **맨 위에** 보여 줘야 할까? 다시 말해, 어떤 페이지가 *좋은* 페이지일까?<br>
The internet has billions of web pages. Search for "puppy" and thousands match. Which one should be shown **at the top**? In other words, which page is a *good* page?


## 첫 번째 생각 : 링크는 투표다<br>First idea : a link is a vote

페이지 X 가 페이지 Y 로 링크를 걸었다면, X 가 "Y 는 볼 만해" 라고 **추천** 한 셈이다. 그러니 *들어오는 링크가 많은* 페이지가 좋은 페이지일 것이다.<br>
If page X links to page Y, then X is **recommending** Y — "Y is worth a look." So a page with *many incoming links* is probably a good page.

하지만 문제가 있다. 가짜 페이지 100 개를 만들어 모두 내 페이지로 링크하면? 링크 수만 세면 속을 수 있다.<br>
But there's a catch. What if I make 100 junk pages all linking to mine? Counting links alone can be fooled.


## 두 번째 생각 : 중요한 페이지의 추천은 더 세게<br>Second idea : a recommendation from an important page counts more

유명하고 믿을 만한 페이지의 추천 한 표가, 이름 없는 가짜 페이지 100 표보다 값지다. 그래서 **"중요한 페이지가 추천할수록 더 중요하다"** 로 규칙을 바꾼다.<br>
One vote from a famous, trusted page is worth more than 100 votes from nameless junk pages. So we change the rule to **"the more important the pages that recommend you, the more important you are."**

그런데 이상하다. 어떤 페이지가 중요한지 알려면, 그 페이지를 추천한 페이지가 중요한지 알아야 하고, 또 그걸 알려면... 빙글빙글 돈다 (순환).<br>
But that's strange: to know if a page is important, we need to know if the pages recommending it are important, and to know *that*... it goes in circles.


## 해결의 실마리 : 일단 똑같이 두고, 반복한다<br>The trick : start everyone equal, then repeat

순환을 한 번에 풀려 하지 말고, **반복** 으로 풀자.<br>
Instead of solving the circle in one shot, solve it by **repetition**.

1. 모든 페이지에 똑같은 점수를 준다.<br>Give every page the same score.
2. 각 페이지는 자기 점수를, 자기가 링크한 페이지들에게 **똑같이 나눠 준다**.<br>Each page hands its score out **equally** to the pages it links to.
3. 2 번을 여러 번 반복한다. 점수는 점점 **안정** 된다.<br>Repeat step 2 many times; the scores **settle down**.

작은 인터넷(페이지 4 개)으로 해 보자.<br>Let's try it on a tiny internet of 4 pages.


In [ ]:
# 작은 인터넷: 누가 누구에게 링크하는가 / a tiny internet: who links to whom
links = {
    'A': ['B', 'C'],   # A 는 B, C 를 추천 / A recommends B and C
    'B': ['C'],        # B 는 C 를 추천 / B recommends C
    'C': ['A', 'B'],   # C 는 A, B 를 추천 / C recommends A and B
    'D': ['C'],        # D 는 C 를 추천 / D recommends C
}
pages = list(links.keys())
print('pages:', pages)
for p in pages:
    print(f'  {p}  ->  {links[p]}')


C 는 A, B, D 세 곳에서 추천을 받는다. 반대로 D 는 아무도 추천하지 않는다. 누가 1 등일지 짐작이 가는가?<br>
C is recommended by three pages (A, B, D). D, on the other hand, is recommended by no one. Can you guess who will come out on top?


### 한 번 돌려 보기<br>One round

모두 점수 $1/4 = 0.25$ 로 시작한다. 각 페이지는 자기 점수를 자기가 링크한 페이지 수만큼 똑같이 나눠 준다.<br>
Everyone starts at $1/4 = 0.25$. Each page splits its score equally among the pages it links to.


In [ ]:
def one_round(score, links):
    '''각 페이지가 자기 점수를 링크한 곳에 똑같이 나눠 준다 / each page hands its score to its links'''
    new_score = {p: 0.0 for p in score}
    for page in score:
        targets = links[page]
        share = score[page] / len(targets)   # 똑같이 나눔 / split equally
        for t in targets:
            new_score[t] += share
    return new_score

score = {p: 1/4 for p in pages}
print('start :', {p: round(score[p], 3) for p in pages})
score = one_round(score, links)
print('round1:', {p: round(score[p], 3) for p in pages})


### 여러 번 반복하기<br>Repeat many times

같은 일을 20 번 반복하면서 점수가 어떻게 변하는지 표로 보자.<br>
Let's repeat the same step 20 times and watch the scores in a table.


In [ ]:
score = {p: 1/4 for p in pages}
history = [dict(score)]

print('round | ' + ' | '.join(f'{p:>5}' for p in pages))
print('-' * 34)
print(f"{0:5d} | " + ' | '.join(f'{score[p]:5.3f}' for p in pages))
for r in range(1, 21):
    score = one_round(score, links)
    history.append(dict(score))
    print(f"{r:5d} | " + ' | '.join(f'{score[p]:5.3f}' for p in pages))


### 그림으로 보기<br>See it as a picture

각 페이지의 점수가 반복에 따라 어떻게 **안정** 되는지 그려 보자.<br>
Let's plot how each page's score **settles** as we repeat.


In [ ]:
import matplotlib.pyplot as plt

rounds = list(range(len(history)))
for p in pages:
    plt.plot(rounds, [h[p] for h in history], marker='o', label=p)

plt.xlabel('round')
plt.ylabel('score')
plt.title('PageRank scores settling down')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# 최종 순위 / the final ranking
ranking = sorted(pages, key=lambda p: score[p], reverse=True)
for rank, p in enumerate(ranking, start=1):
    print(f'{rank}위 / no.{rank}:  page {p}  (score {score[p]:.3f})')


점수가 더 이상 변하지 않고 **안정** 되었다. 이 최종 점수가 바로 페이지의 **순위** 다. 많은 페이지가 — 특히 *중요한* 페이지가 — 가리키는 페이지가 1 등이 된다. 아무도 추천하지 않은 D 는 바닥으로 내려갔다.<br>
The scores stopped changing — they **settled**. These final scores *are* the page **ranking**. The page that many pages — especially *important* ones — point to wins. D, recommended by no one, sank to the bottom.


## 이것이 거듭제곱법이다<br>This is the power method

방금 한 "똑같이 두고 → 점수를 흘려보내고 → 안정될 때까지 반복" 이 과정에는 이름이 있다. 바로 **거듭제곱법(power method)** 이다. 수학자들은 링크 관계를 *행렬* 로 적고, 안정된 점수를 그 행렬의 *고유벡터* 라고 부른다. 같은 이야기를 수학으로 쓴 것이 **거듭제곱법(Power Method)** 과 **구글 페이지랭크(PageRank)** 노트북이다.<br>
That process — "start equal → let the scores flow → repeat until they settle" — has a name: the **power method**. Mathematicians write the link relationships as a *matrix* and call the settled scores its *eigenvector*. The same story written in mathematics is in the **Power Method** and **PageRank** notebooks.

진짜 구글은 한 가지를 더 더한다: 가끔 아무 페이지로나 **임의로 점프** 하게 하는 규칙(감쇠)이다. 그래야 어떤 인터넷에서도 점수가 항상 안정된다. 그 이야기는 **구글 페이지랭크** 노트북에 있다.<br>
The real Google adds one more thing: a rule that *sometimes jumps to a random page* (damping), so the scores always settle on any internet. That story is in the **PageRank** notebook.


## 생각해 볼 질문<br>A question worth sitting with

래리 페이지(Larry Page)와 세르게이 브린(Sergey Brin)은 방금 우리가 손으로 해 본 그 아이디어 — 거듭제곱법 — 하나로 지구에서 가장 큰 기업 중 하나를 세웠다. 그런데 마음에 걸리는 점이 있다. 거듭제곱법을 아는 사람은 수천 명이었다. 그들의 교수님도 알았다. 나도 알았다. 그러나 우리 중 누구도 구글을 만들지 못했다.<br>
Larry Page and Sergey Brin built one of the largest companies on Earth from the idea we just tried by hand — the power method. But here's what should bother you: thousands of people knew the power method. Their professors knew it. I knew it. None of us built Google.

그러니 *방법을 아는 것* 은 분명 드문 일이 아니었다. 그렇다면 드물었던 것은 무엇일까? 나에게도 깔끔한 답은 없다. 다만 몇 가지 짐작을 적어 본다. 여러분도 동의하는지 보자.<br>
So knowing the method clearly wasn't the rare thing. Then what was? I don't have a clean answer — only guesses. See if you agree:

* 어쩌면 그들은 나머지 사람들이 *조용히 포기해 버린* 문제를 알아챘는지도 모른다 — 1998년에 웹을 검색하는 일은 정말 괴로웠다.<br>
  Maybe they *noticed* a problem the rest of us had quietly given up on — that searching the web in 1998 was miserable.
* 어쩌면 드문 능력은 수학이 아니라, *교과서 속 방법이 세상 밖의 어떤 문제에 딱 맞는다는 것을 알아본* 데 있었는지도 모른다.<br>
  Maybe the rare skill wasn't the math, but *recognizing* that a textbook method fit a problem out in the real world.
* 어쩌면 그들은 운도 좋았다 — 알맞은 장소, 알맞은 시기, 알맞은 사람들 — 그리고 그저 멈추지 않았다.<br>
  Maybe they were also lucky — the right place, the right moment, the right people — and simply didn't stop.

나는 마지막 이유가 사람들이 인정하는 것보다 더 중요하다고 생각한다. 그리고 그 사실은 여러분을 *더 나쁘게* 가 아니라 *더 낫게* 만들어 줄 것이다: 그것은 결코 알고리즘을 아는 가장 똑똑한 사람이 되는 일만의 문제가 아니었다.<br>
I think that last one matters more than people admit, and it should make you feel *better*, not worse: it was never only about being the smartest person in the room who knew the algorithm.

*방법을 아는 것* 과 *그것이 어디에 쓰이는지 보는 것* 은 서로 다른 두 능력이다. 학교는 첫 번째를 열심히 훈련시키지만, 두 번째는 대개 스스로 연습해야 한다. 그래서 마지막으로 여러분에게 이 질문을 남긴다.<br>
Knowing a method, and seeing where it matters, are two different skills. School drills the first one hard; the second you mostly have to practise on your own. So here's the question I'd leave you with:

> 내 주위에서 나를 조용히 *불편하게* 하는 어떤 문제가, 내가 이미 알고 있는 어떤 것을 기다리고 있을까?<br>
> What problem around you — one that quietly *bothers* you — might be waiting for something you already know?


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");

